<a href="https://colab.research.google.com/github/gonzH/ifes-redes-neurais-artificiais/blob/main/Trab04_IMDB_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Prencha suas informações: { display-mode: "form" }

nome_completo = 'Hellesandro Gonzaga de Carvalho'  # @param {type: "string"}
matricula     = '20251MCPA0110'  # @param {type: "string"}

data = '2026-07-03'  # @param {type: "date"}
# @markdown ---

## Tarefa:

> Use a classe pipeline da biblioteca transformers do HuggingFace (Behind the pipeline - Hugging Face NLP Course) para analisar o sentimento da base IMBD https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz.

In [4]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz
!rm -r aclImdb/train/unsup
!cat aclImdb/train/pos/4077_10.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  4440k      0  0:00:18  0:00:18 --:--:-- 7199k
I first saw this back in the early 90s on UK TV, i did like it then but i missed the chance to tape it, many years passed but the film always stuck with me and i lost hope of seeing it TV again, the main thing that stuck with me was the end, the hole castle part really touched me, its easy to watch, has a great story, great music, the list goes on and on, its OK me saying how good it is but everyone will take there own best bits away with them once they have seen it, yes the animation is top notch and beautiful to watch, it does show its age in a very few parts but that has now become part of it beauty, i am so glad it has came out on DVD as it is one of my top 10 films of all time. Buy it or rent it just see it, best viewing is at night alone with drin

In [5]:
import os, pathlib, shutil, random
base_dir = pathlib.Path("aclImdb")
train_dir = base_dir / "train"
val_dir = base_dir / "val"
test_dir = base_dir / "test"
for category in ["pos", "neg"]:
  os.makedirs(val_dir / category)
  files = os.listdir(train_dir / category)
  random.shuffle(files)
  num_val_samples = int(0.2 * len(files))
  val_files = files[-num_val_samples:]
  for fname in val_files:
    shutil.move(train_dir / category / fname, val_dir / category / fname)

In [6]:
from tensorflow import keras

batch_size = 32
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


In [7]:
from tensorflow.keras.layers import TextVectorization

max_length = 600
max_tokens=20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length)
text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4)

In [8]:
!pip install transformers

In [ ]:
from transformers import pipeline
import numpy as np

# device=0 para usar gpu
analisador = pipeline("sentiment-analysis", model="lvwerra/distilbert-imdb")

In [17]:
for text_batch, label_batch in test_ds.take(1):
    # converte os tensores para strings e inteiros
    textos = [t.numpy().decode('utf-8') for t in text_batch]
    gabaritos = label_batch.numpy()

    # truncating=True garante que textos mais longos que 512 tokens não quebrem o modelo
    resultados = analisador(textos, truncation=True)

    # exibe os primeiros 5 resultados para comparação
    for i in range(5):
        # positivo = 1, negativo = 0
        label_real = "POSITIVE" if gabaritos[i] == 1 else "NEGATIVE"
        predicao = resultados[i]['label']
        confianca = resultados[i]['score']

        # Cor de destaque se o modelo acertou ou errou
        status = "ACERTOU" if label_real == predicao else "ERROU"

        print(f"Review #{i+1}: {textos[i][:120]}...")
        print(f"Gabarito: {label_real} | Predição: {predicao} ({confianca:.2%}) => {status}")
        print("_" * 60)

Review #1: Isabel has just gone out of jail. She is decided to not return again, but life is difficult for ex-convict, specially if...
Gabarito: POSITIVE | Predição: POSITIVE (99.33%) => ACERTOU
____________________________________________________________
Review #2: I have it on VHS but its not a great copy as I have watched it 2 or 3 times per year since 1999. I am also in fear that ...
Gabarito: POSITIVE | Predição: POSITIVE (93.59%) => ACERTOU
____________________________________________________________
Review #3: This is one of the few movies that was recommended to me as absolutely brilliant, that really is. If you give this movie...
Gabarito: POSITIVE | Predição: POSITIVE (98.80%) => ACERTOU
____________________________________________________________
Review #4: For a movie that was the most seen in its native South Korea for most of 2004, it was a huge disappointment. Shows that ...
Gabarito: NEGATIVE | Predição: NEGATIVE (99.47%) => ACERTOU
_____________________________________